In [4]:
import h5py
import numpy as np
import os
import matplotlib.pyplot as plt
import os
import tensorflow as tf

2026-02-10 15:32:32.693522: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770766352.707402 1182322 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770766352.711737 1182322 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770766352.723622 1182322 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770766352.723635 1182322 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770766352.723637 1182322 computation_placer.cc:177] computation placer alr

In [6]:
# minorized reference
with h5py.File('/pscratch/sd/k/kberard/SCGSR/Data/vo2_1x1x1_opt/density_data/vmc_noJ/density_tot_ref.h5', 'r') as file:
    #print("Keys: %s" % file.keys())
    ref_d = file['density'][:]
#print(ref_d)
print(ref_d.shape)




(116, 116, 72)


In [7]:
import os

# Get the current working directory
current_directory = os.getcwd()

# Print the current working directory
print(current_directory)


/pscratch/sd/k/kberard/SCGSR/EDDA/VO2/Data_Gen


In [8]:
####################################################################################################################################################
def stochastic_density(d,N):
    # poisson model
    #  accurate and fast for all values of N
    # N  = number of MC samples
    assert isinstance(d,np.ndarray)
    assert isinstance(N,(int,float,np.int64,np.float64))
    assert N>0
    ds = np.random.poisson(N*d)/N
    ds*= d.sum()/ds.sum()
    return ds
#end def stochastic_density

####################################################################################################################################################

In [10]:
import numpy as np

# Define parameters
num_train = 1000
num_val = 200
samples_list = [10000000]

def generate_dataset(ref_d, samples, num_train, num_val):
    total_samples = 0
    noise_levels = [samples * (j + 1) for j in range(3)]  # [samples, 2*samples, ..., 5*samples]

    # Generate x_train
    x_train = np.zeros((num_train, 116, 116, 72))
    for i in range(num_train):
        noise_level = np.random.choice(noise_levels)
        total_samples += noise_level
        x_train[i] = stochastic_density(ref_d, noise_level)

    # Generate x_val
    x_val = np.zeros((num_val, 116, 116, 72))
    for i in range(num_val):
        noise_level = np.random.choice(noise_levels)
        total_samples += noise_level
        x_val[i] = stochastic_density(ref_d, noise_level)

    # Generate y_train
    y_train = np.zeros((num_train, 116, 116, 72))
    for i in range(num_train):
        noise_level = np.random.choice(noise_levels) * 100
        y_train[i] = stochastic_density(ref_d, noise_level)

    # Generate y_val
    y_val = np.zeros((num_val, 116, 116, 72))
    for i in range(num_val):
        noise_level = np.random.choice(noise_levels) * 100
        y_val[i] = stochastic_density(ref_d, noise_level)


    return {
        "x_train": x_train,
        "x_val": x_val,
        "y_train": y_train,
        "y_val": y_val,
        "total_samples": total_samples
    }

# Main loop
for samples in samples_list:
    print(f"Generating data for sample scale: {samples}")
    data_dict = generate_dataset(ref_d, samples, num_train, num_val)

    file_name = f"{data_dict['total_samples']}_sample_training_data.npz"
    np.savez("DFT_dat_lev"+file_name,
             x_train=data_dict["x_train"],
             x_val=data_dict["x_val"],
             y_train=data_dict["y_train"],
             y_val=data_dict["y_val"],
             total_samples=data_dict['total_samples'])

    print(f"Saved dataset to {file_name}")


Generating data for sample scale: 10000000
Saved dataset to 24160000000_sample_training_data.npz
